# 3D bosonic toric code at L=4 — phase diagram across h_y planes

One notebook for the L=4 map of the (h_x, h_y, h_z) landscape: what the campaign runs look like,
where each cut puts its transition, and how the boundary moves between h_y planes (2D per plane,
3D overall). All inputs are committed derived JSONs — nothing here needs NetKet or the cluster.

**Inputs**
- `results/phase3d/summary.json` — every landed point of every cut in every h_y plane, with the
  per-cut locators. Regenerate after a pull with
  `python analysis/scripts/phase3d_status.py --export-summary --root results/phase3d --out results/phase3d/summary.json`
  (campaign branch `feat/phase3d-campaign`).
- `results/transitions/hy*_{hx0.2_sweep-hz,hz0.1_sweep-hx}.json` — the two extra L=4 cuts from the
  earlier `hy_cuts_L4` campaign (h_x=0.2 electric, h_z=0.1 magnetic) at h_y = 0, 0.2, 0.4.
- `results/hy_axis_L4/` — the pure h_y axis (h_x=h_z=0), cold points at L=4.

**Conventions (the campaign's, not re-derived here)**
- *Electric cuts* fix h_x and sweep h_z: second-order, located by the loop order parameter
  `O_FM_paratoric` (richards/logistic inflection on the winner curve).
- *Magnetic cuts* fix h_z and sweep h_x: first-order, an `up` chain (from the topological side) and a
  `dn` chain (from the polarized side). Primary locator = the crossing of the two energy branches;
  for h_z ≤ 0.2 (topological → trivial) the membrane order parameter `O_FM_membrane_R1` on the winner
  curve is primary instead, since the 200-step links lag right after the branch destabilises.
  For h_z > 0.2 (trivial → trivial, z-polarized → x-polarized) the primary locator is the **jump of
  ⟨σ^x⟩ on the winner curve** (steepest step between neighbouring links: h_c = bracket midpoint,
  error = half the spacing), cross-checked against the simultaneous jumps of ⟨A_v⟩ and ⟨B_p⟩. A step
  that is neither sharp (≥2× the curve's median slope) nor large (≥0.2) is a *crossover*, not a
  transition. The energy branch crossing is kept as a secondary check only: it compares two separately
  optimized ansätze and is only as good as the worse-converged branch (the h_y=0 dn chains have
  Vscore 0.06–0.12 versus 0.02–0.05 elsewhere, enough to hide a crossing behind a real jump).
- H(h_y) and H(−h_y) are time-reversal partners: E and all diagonal observables are even in h_y, so
  only h_y ≥ 0 is mapped and the 3D plots mirror it.
- Colour encodes the h_y plane throughout (the house plasma-by-L palette is for multi-L figures;
  everything here is L=4).

## 1. Config

In [ ]:
import json, glob, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT     = Path("../..")                          # repo root (cwd = analysis/notebooks)
SUMMARY  = ROOT / "results/phase3d/summary.json"
TRANS    = ROOT / "results/transitions"           # hy_cuts_L4 records (hx=0.2 electric, hz=0.1 magnetic)
HYAXIS   = ROOT / "results/hy_axis_L4/cold/L4"    # pure h_y axis finals
FIGDIR   = Path("figures"); FIGDIR.mkdir(exist_ok=True)   # gitignored; savefig lines stay commented out
L        = 4
BOUND    = -(L**3 + 3*(L-1)**2*L)                 # h=0 OBC stabilizer anchor: -172 at L=4
NSITES   = 3*L**3 - 3*L**2                        # 144 edges
TOPO_HZ_MAX = 0.2                                 # magnetic cuts at hz <= this are topo->trivial (O_FM primary)
EXACT    = {"hz_c(hx=0,hy=0)": 0.193869, "hx_c(hz=0,hy=0)": 1.0}   # thermodynamic-limit anchors (hy=0 only)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

def hy_colors(hys):
    # viridis keyed by h_y (plasma is reserved for L in the multi-L notebooks)
    hys = sorted(hys)
    return {hy: plt.cm.viridis(x) for hy, x in zip(hys, np.linspace(0.0, 0.8, max(len(hys), 2)))}

## 2. Load — points, locators, extra cuts, h_y axis

In [ ]:
S = json.loads(SUMMARY.read_text())
print("summary generated", S["generated"], "| planes", [p["hy"] for p in S["planes"]])

rows, locs = [], []
for pl in S["planes"]:
    hy = pl["hy"]
    for c in pl["cuts"]:
        ff, fv = next(iter(c["fixed"].items()))
        for q in c["points"]:
            if q["L"] != L:
                continue
            rows.append(dict(hy=hy, cut=c["id"], kind=c["kind"], fixed_field=ff, fixed_val=fv, sweep=c["sweep"],
                             **{k: q.get(k) for k in ("h", "branch", "winner", "E0", "E_err", "Vscore", "E_im",
                                                      "O_FM", "O_FM_err", "S2", "S2_err", "sx", "sy", "sz",
                                                      "A_v", "B_p", "diverged", "above_bound", "n_rollbacks",
                                                      "runtime_s")}))
        hs = sorted(q["h"] for q in c["points"] if q["L"] == L)
        base = dict(hy=hy, cut=c["id"], kind=c["kind"], fixed_field=ff, fixed_val=fv, sweep=c["sweep"],
                    h_min=hs[0], h_max=hs[-1], n=len(hs), source="phase3d")
        hc = (c.get("hc") or {}).get(str(L))
        cr = (c.get("crossing") or {}).get(str(L))
        if c["kind"] == "electric":
            locs.append(dict(base, locator="O_FM_paratoric", h_c=hc and hc["h_c"], h_c_err=hc and hc["err"],
                             merged=False, fit=hc and hc["fit"]))
        else:
            topo = fv <= TOPO_HZ_MAX
            if cr is not None:
                locs.append(dict(base, locator="E_crossing", h_c=cr["h_c"], h_c_err=cr["err"], merged=cr["merged"], fit=None))
            if topo and hc is not None:
                locs.append(dict(base, locator="O_FM_membrane", h_c=hc["h_c"], h_c_err=hc["err"], merged=False, fit=hc["fit"]))
            jp = (c.get("jump") or {}).get(str(L))
            if jp is not None:
                locs.append(dict(base, locator="Mx_jump", h_c=jp["h_c"] if jp["ok"] else None, h_c_err=jp["err"] if jp["ok"] else None,
                                 merged=not jp["ok"], fit=None, jump=jp["sx"]["jump"], sharp=jp["sx"]["sharp"], agree=jp["agree"]))

# the earlier hy_cuts_L4 campaign: hx=0.2 electric and hz=0.1 magnetic (membrane-O_FM richards) at hy=0/0.2/0.4
for f in sorted(TRANS.glob("hy*_sweep-*.json")):
    if "@" in f.name:
        continue                                    # '@old' / '@phase3d' lanes are duplicates or the old lane
    d = json.loads(f.read_text())
    ff, fv = next(iter(d["fixed"].items()))
    for r in d["per_L"]:
        if r["L"] != L:
            continue
        kind = "electric" if d["sweep"] == "hz" else "first-order"
        if kind == "first-order" and ff == "hx":
            continue
        locs.append(dict(hy=float(d["hy"]), cut=f"{'electric' if kind=='electric' else 'magnetic'}_{ff}{fv:g}", kind=kind,
                         fixed_field=ff, fixed_val=fv, sweep=d["sweep"], h_min=d["window"][0], h_max=d["window"][1],
                         n=r["n_points"], source="hy_cuts_L4", locator=d["obs"], h_c=r["h_c"], h_c_err=r["h_c_err"],
                         merged=False, fit=None))

pts = pd.DataFrame(rows).sort_values(["hy", "cut", "branch", "h"]).reset_index(drop=True)
NUM = ["h", "E0", "E_err", "Vscore", "E_im", "O_FM", "O_FM_err", "S2", "S2_err", "sx", "sy", "sz", "A_v", "B_p", "runtime_s"]
pts[NUM] = pts[NUM].apply(pd.to_numeric, errors="coerce")      # JSON null -> NaN
loc = pd.DataFrame(locs).sort_values(["hy", "kind", "fixed_val", "locator"]).reset_index(drop=True)
loc[["h_c", "h_c_err"]] = loc[["h_c", "h_c_err"]].apply(pd.to_numeric, errors="coerce")
PLANES = sorted(pts.hy.unique()); COL = hy_colors(PLANES)
print(f"{len(pts)} L={L} points, {len(loc)} locator rows, planes {PLANES}")
loc[["hy", "cut", "locator", "h_c", "h_c_err", "merged", "n", "source"]].round(4)

## 3. Run health per plane

Trust criteria: E below the h=0 bound, `diverged=False`, few rollbacks, Vscore near the plane's floor (≈0.5·h_y² sign-structure cost on top of the real-lane 1e-3…1e-2).

In [ ]:
def health(g):
    return pd.Series({"points": len(g), "diverged": int(g.diverged.sum()), "above_bound": int(g.above_bound.sum()),
                      "rollbacks>0": int((g.n_rollbacks > 0).sum()), "Vscore_med": g.Vscore.median(),
                      "Vscore_max": g.Vscore.max(), "GPU_h": g.runtime_s.sum() / 3600})
H = pts.groupby(["hy", "kind"]).apply(health)
display(H.round(4))

fig, ax = plt.subplots(figsize=(6.2, 3.4))
for hy in PLANES:
    g = pts[pts.hy == hy]
    for kind, m in (("electric", "o"), ("first-order", "s")):
        gg = g[g.kind == kind]
        ax.plot(np.full(len(gg), hy) + (0.012 if kind == "first-order" else -0.012), gg.Vscore, m, ms=4,
                color=COL[hy], alpha=0.6, mec="none")
hy_line = np.linspace(0, max(PLANES) + 0.1, 50)
ax.plot(hy_line, 0.5 * hy_line**2 + 2e-3, "k:", lw=1, label="0.5·h_y² + 2e-3 (sign-structure floor)")
ax.set(xlabel="h_y plane", ylabel="Vscore", yscale="log", title=f"L={L}: Vscore of every landed point (● electric, ■ magnetic)")
ax.legend(fontsize=8, loc="upper left"); fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_vscore_by_plane.png", dpi=300, bbox_inches="tight")

## 4. Electric cuts — what a second-order sweep looks like

Rows: loop order parameter `O_FM_paratoric` (with the campaign's logistic fit and h_c marker), the plaquette Rényi S2, the magnetization ⟨σ^z⟩. Columns: the fixed h_x. Colour: h_y plane.

In [ ]:
def logistic(h, a, b, h0, w):
    return a + b / (1 + np.exp(-(h - h0) / w))

E = pts[(pts.kind == "electric") & pts.winner & ~pts.diverged]
HXS = sorted(E.fixed_val.unique())
fig, axs = plt.subplots(3, len(HXS), figsize=(4.2 * len(HXS), 8.4), sharex="col")
for j, hx in enumerate(HXS):
    for hy in PLANES:
        g = E[(E.fixed_val == hx) & (E.hy == hy)].sort_values("h")
        if g.empty:
            continue
        lab = f"h_y={hy:g}"
        axs[0, j].errorbar(g.h, g.O_FM, g.O_FM_err, fmt="o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        r = loc[(loc.hy == hy) & (loc.cut == f"electric_hx{hx:g}") & (loc.locator == "O_FM_paratoric")]
        if len(r) and r.iloc[0].fit:
            p = r.iloc[0].fit; hh = np.linspace(p["hmin"], p["hmax"], 200)
            axs[0, j].plot(hh, logistic(hh, p["a"], p["b"], p["h0"], p["w"]), "--", lw=1, color=COL[hy], alpha=0.7)
            axs[0, j].axvline(r.iloc[0].h_c, color=COL[hy], lw=0.8, ls=":")
        axs[1, j].errorbar(g.h, g.S2, g.S2_err, fmt="o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        axs[2, j].plot(g.h, g.sz, "o-", ms=4, lw=1.2, color=COL[hy], label=lab)
    axs[0, j].set_title(f"electric cut  h_x = {hx:g}")
    axs[2, j].set(xlabel="h_z")
axs[0, 0].set(ylabel="O_FM (loop)"); axs[1, 0].set(ylabel="S2 (plaquette)"); axs[2, 0].set(ylabel="⟨σ^z⟩")
axs[0, 0].legend(fontsize=8, loc="upper left"); fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_electric_cuts.png", dpi=300, bbox_inches="tight")

## 5. Magnetic cuts — what a first-order sweep looks like

Per fixed h_z (columns): the energy gap between the `up` (topological-side) and `dn` (polarized-side) chains (secondary check: sign change = branch crossing, but its sign is set by the worse-converged branch); the magnetization ⟨σ^x⟩ of the winner branch and its stabilizers — **the primary first-order signal for h_z > 0.2 is their simultaneous jump** (dotted line = M_x-jump locator); for h_z ≤ 0.2 the membrane order parameter is primary (bottom row).

In [ ]:
M = pts[(pts.kind == "first-order") & ~pts.diverged]
HZS = sorted(M.fixed_val.unique())
fig, axs = plt.subplots(4, len(HZS), figsize=(3.6 * len(HZS), 11), sharex="col")
for j, hz in enumerate(HZS):
    for hy in PLANES:
        g = M[(M.fixed_val == hz) & (M.hy == hy)]
        if g.empty:
            continue
        lab = f"h_y={hy:g}"
        up = g[g.branch == "up"].set_index("h").E0; dn = g[g.branch == "dn"].set_index("h").E0
        common = up.index.intersection(dn.index).sort_values()
        axs[0, j].plot(common, (up - dn).loc[common], "o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        w = g[g.winner].sort_values("h")
        axs[1, j].plot(w.h, w.sx, "o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        axs[2, j].plot(w.h, w.A_v, "o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        axs[2, j].plot(w.h, w.B_p, "s--", ms=3, lw=1.0, color=COL[hy], alpha=0.7)
        if hz <= TOPO_HZ_MAX:
            axs[3, j].errorbar(w.h, w.O_FM, w.O_FM_err, fmt="o-", ms=4, lw=1.2, color=COL[hy], label=lab)
        for _, r in loc[(loc.hy == hy) & (loc.cut == f"magnetic_hz{hz:g}") & ~loc.merged & loc.h_c.notna()].iterrows():
            for row in {"E_crossing": [0], "Mx_jump": [1, 2], "O_FM_membrane": [3]}.get(r.locator, []):
                axs[row, j].axvline(r.h_c, color=COL[hy], lw=0.8, ls=":")
    axs[0, j].axhline(0, color="k", lw=0.8)
    axs[0, j].set_title(f"magnetic cut  h_z = {hz:g}")
    axs[3, j].set(xlabel="h_x")
    if hz > TOPO_HZ_MAX:
        axs[3, j].text(0.5, 0.5, "trivial→trivial:\nO_FM not an order parameter", ha="center", va="center",
                       transform=axs[3, j].transAxes, fontsize=8, color="0.5")
axs[0, 0].set(ylabel="E_up − E_dn"); axs[1, 0].set(ylabel="⟨σ^x⟩ (winner)")
axs[2, 0].set(ylabel="⟨A_v⟩ ●, ⟨B_p⟩ ■ (winner)"); axs[3, 0].set(ylabel="O_FM membrane (winner)")
axs[0, 0].legend(fontsize=8, loc="upper left"); fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_magnetic_cuts.png", dpi=300, bbox_inches="tight")

## 6. Boundary table — every locator, every plane

In [ ]:
tab = loc[loc.h_c.notna() | loc.merged].copy()
tab["h_c"] = tab.apply(lambda r: "merged" if r.merged else f"{r.h_c:.3f} ± {r.h_c_err:.3f}", axis=1)
piv = tab.pivot_table(index=["kind", "cut", "locator"], columns="hy", values="h_c", aggfunc="first")
display(piv)

## 7. 2D phase diagrams — one (h_x, h_z) panel per h_y plane

Electric locators are drawn at (fixed h_x, h_z_c) with vertical error bars, magnetic ones at (h_x_c, fixed h_z) with horizontal error bars (▲ membrane O_FM for h_z ≤ 0.2, ■ M_x jump for h_z > 0.2). A cut with no jump (crossover) is shown as its scanned h_x window in grey. The crosses at h_y=0 are the exact thermodynamic-limit anchors.

In [ ]:
MARK = {"O_FM_paratoric": "o", "O_FM_membrane_R1": "^", "O_FM_membrane": "^", "E_crossing": "s", "Mx_jump": "s"}

def draw_plane(ax, hy, color, annotate=True):
    g = loc[loc.hy == hy]
    for _, r in g.iterrows():
        if r.kind == "first-order" and r.fixed_val > TOPO_HZ_MAX and r.locator == "E_crossing":
            continue                                          # secondary check only on trivial->trivial cuts
        if r.merged or not np.isfinite(r.h_c):
            if r.kind == "first-order" and r.locator == "Mx_jump" and r.merged:   # crossover: no jump in the scanned window
                ax.plot([r.h_min, r.h_max], [r.fixed_val] * 2, "-", color="0.75", lw=3, alpha=0.6, zorder=1)
            continue
        if r.kind == "electric":
            ax.errorbar(r.fixed_val, r.h_c, yerr=r.h_c_err, fmt=MARK[r.locator], color=color, mec="k", mew=0.4,
                        ms=7, ecolor="0.5", capsize=2, zorder=3)
        else:
            ax.errorbar(r.h_c, r.fixed_val, xerr=r.h_c_err, fmt=MARK[r.locator], color=color, mec="k", mew=0.4,
                        ms=7, ecolor="0.5", capsize=2, zorder=3)
    # topological envelope: electric points (h_x ascending) then the membrane-O_FM points at h_z <= 0.2
    # (h_z descending) -- the closed boundary of the topological phase in this plane
    e = g[(g.kind == "electric") & g.h_c.notna()].sort_values("fixed_val")
    t = g[(g.kind == "first-order") & (g.fixed_val <= TOPO_HZ_MAX) & g.locator.str.startswith("O_FM")]
    t = t.sort_values("fixed_val", ascending=False)
    ax.plot(list(e.fixed_val) + list(t.h_c), list(e.h_c) + list(t.fixed_val), "-", color=color, lw=1.2, alpha=0.6, zorder=2)
    # trivial->trivial first-order line (M_x jumps at h_z >= 0.4), continuing from the electric end
    v = g[(g.kind == "first-order") & (g.fixed_val > TOPO_HZ_MAX) & (g.locator == "Mx_jump") & g.h_c.notna() & ~g.merged]
    v = v.sort_values("fixed_val")
    if len(v) and len(e):
        ax.plot([e.fixed_val.iloc[-1]] + list(v.h_c), [e.h_c.iloc[-1]] + list(v.fixed_val), "--", color=color, lw=1, alpha=0.6, zorder=2)
    if annotate and hy == 0:
        ax.plot(0, EXACT["hz_c(hx=0,hy=0)"], "k+", ms=10, mew=1.2, zorder=4)
        ax.plot(EXACT["hx_c(hz=0,hy=0)"], 0, "k+", ms=10, mew=1.2, zorder=4)
    ax.text(0.25, 0.08, "topological", fontsize=9, color="0.4", ha="center")
    ax.text(1.35, 0.9, "trivial", fontsize=9, color="0.4", ha="center")

fig, axs = plt.subplots(1, len(PLANES), figsize=(4.6 * len(PLANES), 4.4), sharex=True, sharey=True)
for ax, hy in zip(np.atleast_1d(axs), PLANES):
    draw_plane(ax, hy, COL[hy])
    ax.set(title=f"h_y = {hy:g}", xlabel="h_x", xlim=(-0.05, 1.75), ylim=(-0.05, 1.1))
np.atleast_1d(axs)[0].set(ylabel="h_z")
handles = [Line2D([], [], marker="o", color="0.3", ls="", mec="k", label="electric: O_FM loop (2nd order)"),
           Line2D([], [], marker="s", color="0.3", ls="", mec="k", label="magnetic: M_x jump, stabilizers agree (hz>0.2)"),
           Line2D([], [], marker="^", color="0.3", ls="", mec="k", label="magnetic: membrane O_FM (hz≤0.2)"),
           Line2D([], [], color="0.75", lw=3, label="scanned window, no jump (crossover)"),
           Line2D([], [], color="0.3", lw=1.2, label="topological envelope"),
           Line2D([], [], color="0.3", lw=1, ls="--", label="1st-order line, trivial→trivial"),
           Line2D([], [], marker="+", color="k", ls="", ms=10, label="exact L→∞ anchors (h_y=0)")]
np.atleast_1d(axs)[0].legend(handles=handles, fontsize=7, loc="upper right")
fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_planes_2d.png", dpi=300, bbox_inches="tight")

### 7b. All planes overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for hy in PLANES:
    draw_plane(ax, hy, COL[hy], annotate=(hy == 0))
ax.set(xlabel="h_x", ylabel="h_z", xlim=(-0.05, 1.75), ylim=(-0.05, 1.1), title=f"L={L} boundary, colour = h_y plane")
ax.legend(handles=[Line2D([], [], color=COL[hy], lw=3, label=f"h_y = {hy:g}") for hy in PLANES], fontsize=8, loc="upper right")
fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_planes_overlay.png", dpi=300, bbox_inches="tight")

## 8. The pure h_y axis (h_x = h_z = 0)

Cold points from `results/hy_axis_L4`, two seeds where available; the lower-energy seed is the winner. The sign-structure cost makes Vscore ≈ 0.5·h_y² the floor here, so the energy is the trustworthy signal: a first-order jump shows up as an energy kink plus a ⟨σ^y⟩ jump.

In [ ]:
ax_rows = []
for f in sorted(HYAXIS.glob("*.json")):
    if f.name.endswith((".curve.json", ".finaleval_electric.json", ".snapshots.json")):
        continue
    d = json.loads(f.read_text()); c, o = d["config"], d["observables"]
    ax_rows.append(dict(hy=c["hy"], seed=int(d["name"].endswith("_s1")), E0=o["E0"], Vscore=o["Vscore"],
                        sy=o.get("sy_mean"), diverged=d["diverged"]))
A = pd.DataFrame(ax_rows)
A = A[A.hy >= 0].sort_values(["hy", "seed"])
Aw = A.loc[A.groupby("hy").E0.idxmin()]                       # winner seed per hy
jump = Aw.assign(dE=np.gradient(Aw.E0, Aw.hy))
display(Aw.round(4).T)

fig, axs = plt.subplots(1, 3, figsize=(12.5, 3.4))
axs[0].plot(A.hy, A.E0, "o", ms=4, color="0.6", label="all seeds")
axs[0].plot(Aw.hy, Aw.E0, "o-", ms=4, lw=1.2, color=COL[PLANES[-1]], label="winner")
axs[0].axhline(BOUND, color="k", ls=":", lw=1, label="h=0 bound")
axs[0].set(xlabel="h_y", ylabel="E"); axs[0].legend(fontsize=8, loc="lower left")
axs[1].plot(A.hy, A.sy, "o", ms=4, color="0.6"); axs[1].plot(Aw.hy, Aw.sy, "o-", ms=4, lw=1.2, color=COL[PLANES[-1]])
axs[1].set(xlabel="h_y", ylabel="⟨σ^y⟩ (winner)")
axs[2].plot(A.hy, A.Vscore, "o", ms=4, color="0.6"); axs[2].plot(Aw.hy, Aw.Vscore, "o-", ms=4, lw=1.2, color=COL[PLANES[-1]])
hh = np.linspace(0.05, A.hy.max(), 50); axs[2].plot(hh, 0.5 * hh**2, "k:", lw=1, label="0.5·h_y²")
axs[2].set(xlabel="h_y", ylabel="Vscore", yscale="log"); axs[2].legend(fontsize=8)
fig.suptitle(f"pure h_y axis, L={L} cold points", y=1.02); fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_hy_axis.png", dpi=300, bbox_inches="tight")

# first-order jump on the axis. The raw winner grid brackets it by the largest ⟨σ^y⟩ jump, but a cold point
# just below the jump can sit on the metastable topological branch (its E is smooth from below while the
# polarized branch has slope dE/dh_y = -N⟨σ^y⟩ ≈ -135). So also extrapolate the two branches and cross them:
d_sy = np.diff(Aw.sy.to_numpy()); k = int(np.nanargmax(np.abs(d_sy)))
HY_AXIS_BRACKET = (float(Aw.hy.iloc[k]), float(Aw.hy.iloc[k + 1]))
topo = Aw[Aw.hy <= HY_AXIS_BRACKET[0]]; pol = Aw[Aw.hy >= HY_AXIS_BRACKET[1]]
p_topo = np.polyfit(topo.hy, topo.E0, 2); p_pol = np.polyfit(pol.hy, pol.E0, 1)
hh = np.linspace(HY_AXIS_BRACKET[0] - 0.3, HY_AXIS_BRACKET[1], 400)
gap = np.polyval(p_topo, hh) - np.polyval(p_pol, hh)
HY_AXIS_CROSS = float(hh[np.argmin(np.abs(gap))]) if np.any(np.sign(gap[:-1]) != np.sign(gap[1:])) else np.mean(HY_AXIS_BRACKET)
print(f"h_y-axis jump: raw bracket {HY_AXIS_BRACKET} (Δ⟨σ^y⟩ = {d_sy[k]:.3f}); branch-crossing estimate h_y,c ≈ {HY_AXIS_CROSS:.3f}"
      f" (polarized slope {p_pol[0]:.1f} vs −N = {-NSITES})")

## 9. 3D view — the boundary points in (h_x, h_y, h_z)

Every located transition from every plane, mirrored to −h_y by time reversal (lighter), plus the h_y-axis bracket midpoint. Translucent surfaces are triangulations of the electric (second-order) and magnetic (first-order) point families — a guide to the eye at three planes, not a fit.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

B = loc[loc.h_c.notna() & ~loc.merged].copy()
B["hx"] = np.where(B.kind == "electric", B.fixed_val, B.h_c)
B["hz"] = np.where(B.kind == "electric", B.h_c, B.fixed_val)
# primary locator per (hy, cut): membrane O_FM at hz <= 0.2, energy crossing above (the campaign convention)
B["primary"] = np.where(B.kind == "electric", True,
                np.where(B.fixed_val <= TOPO_HZ_MAX, B.locator.str.startswith("O_FM"), B.locator == "Mx_jump"))
B = B[B.primary].drop_duplicates(["hy", "kind", "fixed_val"])
B["family"] = np.where((B.kind == "first-order") & (B.fixed_val > TOPO_HZ_MAX), "1st-order line (trivial→trivial)",
              np.where(B.kind == "electric", "topological envelope: 2nd order", "topological envelope: 1st order"))
axis_pt = pd.DataFrame([dict(hx=0.0, hy=HY_AXIS_CROSS, hz=0.0)])
FCOL = {"topological envelope: 2nd order": "#2a7fd5", "topological envelope: 1st order": "#d5542a",
        "1st-order line (trivial→trivial)": "#8c8c8c"}
env = B[B.family.str.startswith("topological")]
tt = B[~B.family.str.startswith("topological")]

fig = plt.figure(figsize=(13, 5.2))
for i, (elev, azim) in enumerate([(22, -60), (28, 30)]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    for sgn, alpha in ((1, 1.0), (-1, 0.25)):
        for fam, g in B.groupby("family"):
            ax.scatter(g.hx, sgn * g.hy, g.hz, s=28, color=FCOL[fam], alpha=alpha, edgecolor="k", lw=0.3,
                       label=fam if sgn == 1 else None)
        if len(env) >= 3 and env.hy.nunique() >= 2:      # one surface for the whole topological envelope
            ax.plot_trisurf(env.hx, sgn * env.hy, env.hz, color="#2a7fd5", alpha=0.12 if sgn == 1 else 0.05, lw=0)
        for hy_, g in tt.groupby("hy"):                    # the trivial-trivial line, per plane
            g = g.sort_values("hz")
            ax.plot(g.hx, sgn * g.hy, g.hz, "--", color="0.5", lw=1, alpha=alpha)
        ax.scatter(axis_pt.hx, sgn * axis_pt.hy, axis_pt.hz, s=70, marker="*", color="k", alpha=alpha,
                   label="h_y-axis branch crossing" if sgn == 1 else None)
    ax.plot([0, 0], [-axis_pt.hy.iloc[0], axis_pt.hy.iloc[0]], [0, 0], "k:", lw=1)
    ax.set(xlabel="h_x", ylabel="h_y", zlabel="h_z", xlim=(0, 1.6), zlim=(0, 1.1))
    ax.view_init(elev=elev, azim=azim)
    if i == 0:
        ax.legend(fontsize=7, loc="upper left")
fig.suptitle(f"L={L} phase boundary of the 3D bosonic toric code (topological phase encloses the origin)", y=0.98)
fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_boundary_3d.png", dpi=300, bbox_inches="tight")

### 9b. How the boundary moves with h_y

Each located cut against the plane's h_y. Flat = the h_y field does not move that part of the boundary at L=4; a slope tells where the next planes will be interesting.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for kind, ax, ttl in (("electric", axs[0], "electric: h_z,c vs h_y (label = fixed h_x)"),
                      ("first-order", axs[1], "magnetic: h_x,c vs h_y (label = fixed h_z)")):
    g = B[B.kind == kind]
    for fv, gg in g.groupby("fixed_val"):
        gg = gg.sort_values("hy")
        ax.errorbar(gg.hy, gg.h_c, gg.h_c_err, fmt="o-", ms=4, lw=1.2, capsize=2, label=f"{gg.fixed_field.iloc[0]}={fv:g}")
    ax.set(xlabel="h_y", ylabel="h_z,c" if kind == "electric" else "h_x,c", title=ttl)
    ax.legend(fontsize=8, loc="best")
fig.tight_layout(); plt.show()
# fig.savefig(FIGDIR / "phase3d_L4_hc_vs_hy.png", dpi=300, bbox_inches="tight")

## 10. Reading the map (2026-09-17, three planes) and what to run next

**What the three planes say**

- **Electric (second-order) sheet** h_z,c(h_x, h_y): flat in h_x up to 0.5 (≈0.29 at h_y=0), rising to ≈0.34–0.35
  at h_x=0.8; nearly h_y-independent through h_y=0.2 and only ~0.02 lower at h_y=0.4. The L=4 value at
  h_x=0 sits 0.10 above the exact L→∞ anchor 0.194 (the L=4/5/6 FSS at h_y=0 does extrapolate to it).
- **Topological envelope, first-order part** (membrane O_FM at h_z ≤ 0.2): h_x,c ≈ 0.80–0.83 in all three
  planes, ~0.02 lower at h_y=0.4. The h_z=0 energy crossing at h_y=0 (0.974) is the 200-step link-lag artifact
  documented in the campaign log, not a locator disagreement.
- **Trivial→trivial first-order line** (h_z ≥ 0.4, M_x jump with ⟨A_v⟩/⟨B_p⟩ agreeing in every case):
  h_z=0.4 jumps at 0.888(13) / 0.825(25) / 0.825(25) (Δ⟨σ^x⟩ = 0.31–0.41 within one link); **h_z=0.7 jumps
  at 1.20(5) in all three planes** (Δ ≈ 0.27, but the bracket [1.15, 1.25] is a 0.1 gap in the grid — links at
  1.2 would halve the error); **h_z=1.0 is a crossover** in every plane (largest step ≤ 0.18 spread over
  0.1–0.25). So the first-order line's critical endpoint lies between h_z=0.7 and h_z=1.0. The energy branch
  crossing agrees where both chains are well converged (h_z=0.4 at h_y=0.2/0.4) and fails where the dn chain
  is poor (h_y=0: Vscore 0.06–0.12, "merged" at h_z=0.7 despite the jump) — it stays a secondary check.
- **h_y axis**: first-order at L=4, branch-crossing estimate h_y,c ≈ 1.16 (the h_y=1.2 cold point sits on the
  metastable topological branch). So the topological region closes in h_y between the h_y=1.0 and 1.2 planes,
  and every plane up to h_y≈1.0 still has a finite topological lobe to map.
- **Cost**: ≈25 GPU-h per plane at L=4 (9 electric + 15 magnetic); complex-lane L=4 is ≈3.1 s/step.
- **Quality caveat**: Vscore floor 0.5·h_y² (0.32 at h_y=0.8, 0.5 at h_y=1.0). The h_y axis shows the
  ansatz still finds the right branch there, but O_FM/S2 get noisier — check §3/§4 for each new plane.

**Run plan (L=4 only, h_y ≥ 0 by time-reversal symmetry)**

| priority | what | jobs | GPU-h | why |
|---|---|---|---|---|
| 1 | h_z=0.7 links at h_x=1.2 (up and dn), planes 0/0.2/0.4 | 6 | ~2 | resolve the jump the current grid straddles |
| 2 | plane **h_y=0.6**, all 8 cuts (planner defaults) | ~71 | ~25 | next slice; sign cost still moderate (0.18) |
| 3 | plane **h_y=0.8**, all 8 cuts | ~71 | ~25 | envelope shrinking? |
| 4 | new magnetic cut **h_z=0.85** in every plane | ~14/plane | ~5/plane | bracket the first-order line's endpoint |
| 5 | plane **h_y=1.0**, all 8 cuts | ~71 | ~25 | last plane before the lobe closes (h_y,c ≈ 1.16 on the axis) |
| 6 | h_y-axis up/dn chain h_y ∈ [0.9, 1.4] at h_x=h_z=0 | ~16 | ~6 | an error bar on the axis point (needs a sweep-h_y cut type) |
| later | plane h_y=1.2 (expected fully trivial), electric grid rebalancing where the L=4 fit lands skewed | — | — | closure check |

Planner changes needed: add the planes to `HY_VALUES` with `_DHY` entries (the empirical shift is ≈ −0.14·h_y²:
−0.05/−0.09/−0.14 at 0.6/0.8/1.0), add 0.85 to `TAIL_HZ` with anchors (0.6, 1.6), and a chain insert at h_x=1.2
for h_z=0.7. Everything else (windows, recentring, dedup manifests, scrontab drivers) is generic over h_y.